In [1]:
import pandas as pd
INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"

def leer_fasta(path):
    s={}; h=None; b=[]
    with open(path) as f:
        for line in f:
            line=line.rstrip("\n")
            if line.startswith(">"):
                if h: s[h]="".join(b)
                h=line[1:].split()[0]; b=[]
            else: b.append(line.strip())
        if h: s[h]="".join(b)
    return s

genes = leer_fasta(f"{INTERIM}/muestra_genes_completos.fasta")
READ = 150
n_reads = 0
with open(f"{INTERIM}/muestra_reads150.fasta", "w") as out:
    for gid, seq in genes.items():
        seq = seq.upper()
        for i in range(len(seq)//READ):          # ventanas de 150 no solapadas
            out.write(f">{gid}__r{i}\n{seq[i*READ:(i+1)*READ]}\n")
            n_reads += 1
print(f"Lecturas de 150 nt generadas: {n_reads}")

Lecturas de 150 nt generadas: 11392


In [3]:
import pandas as pd, numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# === RUTAS (ajusta si difieren) ===
ARGNET_OUT = "/home/r4gbi/ARGNet/results/argnet_reads150.txt"
LABELS = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim/muestra_genes_labels.tsv"

# ---------- 1) Salida ARGNet-s (una fila por READ) ----------
reads = pd.read_csv(ARGNET_OUT, sep=None, engine="python")
print("Columnas crudas:", list(reads.columns))
print(reads.head(3), "\n")

# columna de predicción = la que contiene valores ARG / non-ARG
col_pred = next(c for c in reads.columns
                if reads[c].astype(str).str.strip().str.upper().isin(["ARG","NON-ARG"]).any())
col_id   = reads.columns[0] if reads.columns[0] != col_pred else reads.columns[1]

reads["pred_bin"] = (reads[col_pred].astype(str).str.strip().str.upper() == "ARG").astype(int)
reads["gid"]      = reads[col_id].astype(str).str.split("__r").str[0]   # quita el sufijo __r{i}

print(f"Reads evaluadas: {len(reads)} | Genes representados: {reads['gid'].nunique()}")

# ---------- 2) Ground truth (nivel gen) ----------
truth = pd.read_csv(LABELS, sep="\t")
gid_col = truth.columns[0]
lab_col = next(c for c in truth.columns[1:]
               if pd.to_numeric(truth[c], errors="coerce").dropna().isin([0,1]).all())
truth = truth[[gid_col, lab_col]].rename(columns={gid_col:"gid", lab_col:"y_true"})
truth["y_true"] = truth["y_true"].astype(int)
print(f"Genes en referencia: {len(truth)} "
      f"(pos={truth['y_true'].sum()}, neg={(truth['y_true']==0).sum()})\n")

def reporte(y_true, y_pred, titulo):
    print(f"=== {titulo} ===")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred,
          target_names=["no-ARG (0)","ARG (1)"], digits=3))

# ---------- 3a) Nivel READ (cada read hereda la etiqueta de su gen) ----------
rd = reads.merge(truth, on="gid", how="inner")
reporte(rd["y_true"], rd["pred_bin"],
        f"ARGNet-s | NIVEL READ ({len(rd)} reads)")

# ---------- 3b) Nivel GEN - regla OR (cualquier read ARG) ----------
g_or = reads.groupby("gid")["pred_bin"].max().rename("y_pred").reset_index()
d_or = truth.merge(g_or, on="gid", how="left")
sin_reads = d_or["y_pred"].isna().sum()
print(f"Genes sin NINGUNA read en la salida (se cuentan como no-ARG): {sin_reads}")
d_or["y_pred"] = d_or["y_pred"].fillna(0).astype(int)
reporte(d_or["y_true"], d_or["y_pred"],
        "ARGNet-s | NIVEL GEN - regla OR (cualquier read ARG)")

# ---------- 3c) Nivel GEN - regla mayoría (>50% de reads ARG) ----------
g_maj = (reads.groupby("gid")["pred_bin"].mean() >= 0.5).astype(int).rename("y_pred").reset_index()
d_maj = truth.merge(g_maj, on="gid", how="left")
d_maj["y_pred"] = d_maj["y_pred"].fillna(0).astype(int)
reporte(d_maj["y_true"], d_maj["y_pred"],
        "ARGNet-s | NIVEL GEN - regla MAYORIA (>50% reads ARG)")

Columnas crudas: ['test_id', 'ARG_prediction', 'resistance_category', 'probability']
                                             test_id ARG_prediction  \
0  CARD|gb|MN313890.1|+|0-906|ARO:3005060|OXA-837...            ARG   
1  CARD|gb|MN313890.1|+|0-906|ARO:3005060|OXA-837...            ARG   
2  CARD|gb|MN313890.1|+|0-906|ARO:3005060|OXA-837...            ARG   

  resistance_category  probability  
0         beta-lactam     0.996720  
1         beta-lactam     0.549896  
2                 MLS     0.863977   

Reads evaluadas: 11392 | Genes representados: 1977
Genes en referencia: 2000 (pos=1000, neg=1000)

=== ARGNet-s | NIVEL READ (11392 reads) ===
[[5894  117]
 [2724 2657]]
              precision    recall  f1-score   support

  no-ARG (0)      0.684     0.981     0.806      6011
     ARG (1)      0.958     0.494     0.652      5381

    accuracy                          0.751     11392
   macro avg      0.821     0.737     0.729     11392
weighted avg      0.813     0.751     

In [5]:
# 1) vocabulario de categorías de ARGNet
print("=== Categorías que predice ARGNet ===")
print(reads.groupby("ARG_prediction")["resistance_category"].value_counts(), "\n")

# 2) un registro por gen + base de datos de origen (primer token del header)
g = reads.drop_duplicates("gid").copy()
g["fuente"] = g["gid"].str.split("|").str[0]
print("=== Genes por base de datos de origen ===")
print(g["fuente"].value_counts(), "\n")

# 3) muestra de headers por fuente (para diseñar el extractor de clase verdadera)
for f, sub in g.groupby("fuente"):
    print(f"--- {f}  (n={len(sub)}) ---")
    for h in sub["gid"].head(4):
        print("   ", h[:140])
    print()

=== Categorías que predice ARGNet ===
ARG_prediction  resistance_category
ARG             bacitracin             1157
                beta-lactam             731
                multidrug               361
                MLS                     266
                aminoglycoside           65
                tetracycline             55
                glycopeptide             53
                trimethoprim             40
                quinolone                10
                chloramphenicol          10
                mupirocin                 9
                peptide                   6
                qa_compound               4
                fosfomycin                4
                ethambutol                3
Name: count, dtype: int64 

=== Genes por base de datos de origen ===
fuente
ResFinderFG        475
CARD               328
GCF_000412675.1    209
GCF_000008865.2    175
GCF_000009045.1    175
GCF_000025985.1    169
GCF_000005845.2    153
MEGARes            112
GCF_0

In [4]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def fila(nombre, y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], average=None)
    return (nombre, r[0], p[0], f[0], accuracy_score(y_true, y_pred))

resumen = [
    fila("ARGNet-s  read-level", rd["y_true"],    rd["pred_bin"]),
    fila("ARGNet-s  gen OR",     d_or["y_true"],   d_or["y_pred"]),
    fila("ARGNet-s  gen mayoria",d_maj["y_true"],  d_maj["y_pred"]),
]

import pandas as pd
tabla = pd.DataFrame(resumen,
        columns=["metodo","recall_ARG","precision_ARG","f1_ARG","accuracy"]).round(3)
print(tabla.to_string(index=False))
print("\nReferencia argnet-l (genes completos): recall_ARG = 0.553")
print(f"Genes <150nt no evaluables por argnet-s: {2000 - reads['gid'].nunique()}")

               metodo  recall_ARG  precision_ARG  f1_ARG  accuracy
 ARGNet-s  read-level       0.494          0.958   0.652     0.751
     ARGNet-s  gen OR       0.589          0.882   0.706     0.755
ARGNet-s  gen mayoria       0.475          0.983   0.641     0.734

Referencia argnet-l (genes completos): recall_ARG = 0.553
Genes <150nt no evaluables por argnet-s: 23
